# GeoFlood-Agent — SegFormer Training Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadBilal0021/GeoFloodAgent/blob/main/notebooks/01_train_segformer.ipynb)

Fine-tunes SegFormer-B2 on Sen1Floods11 for binary flood segmentation.

**Run order:** Setup -> Dataset -> Model -> Train -> Evaluate -> Push to Hub

**Requires:** GPU runtime (Runtime -> Change runtime type -> T4 GPU)


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/geoflood"
DATA_DIR = f"{DRIVE_ROOT}/sen1floods11"
CHECKPOINT_DIR = f"{DRIVE_ROOT}/checkpoints"

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Data dir exists: {os.path.exists(DATA_DIR)}")
print(f"S2Hand tiles: {len(os.listdir(f'{DATA_DIR}/S2Hand'))}")
print(f"LabelHand tiles: {len(os.listdir(f'{DATA_DIR}/LabelHand'))}")

In [ ]:
!pip install -q transformers accelerate rasterio huggingface_hub scikit-learn

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "No GPU detected - go to Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Requires HF_TOKEN saved in Colab Secrets (key icon, left sidebar)
login(token=userdata.get("HF_TOKEN"))
print("HuggingFace login OK")

## 2. Dataset

Sen1Floods11 hand-labeled split. Labels: `-1` = no-data/cloud (excluded from loss),
`0` = no flood, `1` = flood.

We build the train/val/test split ourselves **by event**, not by individual tile,
to prevent spatial leakage — tiles from the same flood event must stay together
in the same split, otherwise the model can "cheat" by memorizing a location
instead of learning general flood patterns.

In [ ]:
import glob
from collections import defaultdict
import random

s2_files = sorted(glob.glob(f"{DATA_DIR}/S2Hand/*.tif"))
lb_files = sorted(glob.glob(f"{DATA_DIR}/LabelHand/*.tif"))
assert len(s2_files) == len(lb_files), "S2 and Label tile counts do not match"

# Group tiles by country/event prefix (e.g. "Pakistan", "Spain")
by_event = defaultdict(list)
for s2, lb in zip(s2_files, lb_files):
    event = os.path.basename(s2).split("_")[0]
    by_event[event].append((s2, lb))

events = sorted(by_event.keys())
print(f"Events found: {events}")
for e in events:
    print(f"  {e}: {len(by_event[e])} tiles")

In [ ]:
# Event-level split - held-out events for val/test so the model is
# evaluated on geography it has never seen during training
random.seed(42)
shuffled_events = events.copy()
random.shuffle(shuffled_events)

n_val_events = max(1, len(events) // 6)
n_test_events = max(1, len(events) // 6)

test_events = shuffled_events[:n_test_events]
val_events = shuffled_events[n_test_events:n_test_events + n_val_events]
train_events = shuffled_events[n_test_events + n_val_events:]

train_pairs = [p for e in train_events for p in by_event[e]]
val_pairs   = [p for e in val_events   for p in by_event[e]]
test_pairs  = [p for e in test_events  for p in by_event[e]]

print(f"Train: {len(train_pairs)} tiles from {train_events}")
print(f"Val:   {len(val_pairs)} tiles from {val_events}")
print(f"Test:  {len(test_pairs)} tiles from {test_events}")

In [ ]:
import rasterio
import numpy as np
import torch
from torch.utils.data import Dataset

# Sentinel-2 band indices (0-indexed) for RGB - the 3-channel input SegFormer expects
RGB_BANDS = [3, 2, 1]  # Red, Green, Blue

class FloodDataset(Dataset):
    def __init__(self, pairs, processor, augment=False):
        self.pairs = pairs
        self.processor = processor
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        s2_path, lb_path = self.pairs[idx]

        with rasterio.open(s2_path) as src:
            img = src.read(RGB_BANDS).astype(np.float32)  # (3, H, W)

        with rasterio.open(lb_path) as src:
            label = src.read(1).astype(np.int64)  # (H, W), values -1/0/1

        # Normalize Sentinel-2 reflectance into 0-255 visual range
        img = np.clip(img / 3000.0 * 255.0, 0, 255).astype(np.uint8)
        img = img.transpose(1, 2, 0)  # (H, W, 3) for the processor

        if self.augment:
            if random.random() > 0.5:
                img = np.ascontiguousarray(img[:, ::-1, :])
                label = np.ascontiguousarray(label[:, ::-1])
            if random.random() > 0.5:
                img = np.ascontiguousarray(img[::-1, :, :])
                label = np.ascontiguousarray(label[::-1, :])

        encoded = self.processor(images=img, return_tensors="pt")
        pixel_values = encoded["pixel_values"].squeeze(0)

        # Resize label to match SegFormer's output resolution (1/4 of input)
        label_tensor = torch.from_numpy(label).unsqueeze(0).unsqueeze(0).float()
        label_resized = torch.nn.functional.interpolate(
            label_tensor, size=(128, 128), mode="nearest"
        ).squeeze().long()

        # Remap -1 (no-data) to 255 so cross_entropy's ignore_index can exclude it
        label_resized[label_resized == -1] = 255

        return {"pixel_values": pixel_values, "labels": label_resized}

print("FloodDataset class defined")

In [ ]:
from transformers import SegformerImageProcessor

processor = SegformerImageProcessor(
    do_resize=True, size={"height": 512, "width": 512},
    do_normalize=True,
)

train_ds = FloodDataset(train_pairs, processor, augment=True)
val_ds   = FloodDataset(val_pairs, processor, augment=False)
test_ds  = FloodDataset(test_pairs, processor, augment=False)

# Quick sanity check on one sample
sample = train_ds[0]
print(f"pixel_values shape: {sample['pixel_values'].shape}")
print(f"labels shape: {sample['labels'].shape}")
print(f"unique label values: {torch.unique(sample['labels'])}")
print("Dataset sanity check PASSED")

## 3. Model

In [ ]:
from transformers import SegformerForSemanticSegmentation

id2label = {0: "no_flood", 1: "flood"}
label2id = {v: k for k, v in id2label.items()}

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b2",
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
).cuda()

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded - {n_params/1e6:.1f}M parameters")

## 4. Training Loop

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 8
EPOCHS = 30
LR = 6e-5
IGNORE_INDEX = 255

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda step: min(1.0, step / max(1, warmup_steps))
)

print(f"Train batches/epoch: {len(train_loader)}")
print(f"Val batches/epoch: {len(val_loader)}")
print(f"Total training steps: {total_steps}")

In [ ]:
def compute_iou(preds, labels, ignore_index=255):
    """Binary IoU for the flood class, excluding ignore_index pixels."""
    valid = labels != ignore_index
    preds_v = preds[valid]
    labels_v = labels[valid]

    intersection = ((preds_v == 1) & (labels_v == 1)).sum().item()
    union = ((preds_v == 1) | (labels_v == 1)).sum().item()

    if union == 0:
        return None  # no flood pixels in this batch, skip
    return intersection / union

print("IoU function defined")

In [ ]:
import time

best_val_iou = 0.0
history = {"train_loss": [], "val_loss": [], "val_iou": []}

# Resume from checkpoint if one exists (handles Colab session interruptions)
checkpoint_path = f"{CHECKPOINT_DIR}/latest_checkpoint.pt"
start_epoch = 0

if os.path.exists(checkpoint_path):
    print("Found existing checkpoint - resuming training")
    ckpt = torch.load(checkpoint_path, map_location="cuda")
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_val_iou = ckpt["best_val_iou"]
    history = ckpt["history"]
    print(f"Resumed at epoch {start_epoch}, best IoU so far: {best_val_iou:.4f}")
else:
    print("No checkpoint found - starting fresh")

In [ ]:
for epoch in range(start_epoch, EPOCHS):
    # Train
    model.train()
    epoch_loss = 0.0
    t0 = time.time()

    for batch in train_loader:
        pixel_values = batch["pixel_values"].cuda()
        labels = batch["labels"].cuda()

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()

    avg_train_loss = epoch_loss / len(train_loader)

    # Validate
    model.eval()
    val_loss = 0.0
    val_ious = []

    with torch.no_grad():
        for batch in val_loader:
            pixel_values = batch["pixel_values"].cuda()
            labels = batch["labels"].cuda()

            outputs = model(pixel_values=pixel_values, labels=labels)
            val_loss += outputs.loss.item()

            preds = outputs.logits.argmax(dim=1)
            iou = compute_iou(preds, labels, ignore_index=IGNORE_INDEX)
            if iou is not None:
                val_ious.append(iou)

    avg_val_loss = val_loss / len(val_loader)
    avg_val_iou = sum(val_ious) / len(val_ious) if val_ious else 0.0

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["val_iou"].append(avg_val_iou)

    elapsed = time.time() - t0
    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"train_loss={avg_train_loss:.4f} | "
          f"val_loss={avg_val_loss:.4f} | "
          f"val_iou={avg_val_iou:.4f} | "
          f"{elapsed:.1f}s")

    # Save checkpoint every epoch (resumable on Colab disconnect)
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_iou": best_val_iou,
        "history": history,
    }, checkpoint_path)

    # Save best model separately
    if avg_val_iou > best_val_iou:
        best_val_iou = avg_val_iou
        model.save_pretrained(f"{CHECKPOINT_DIR}/best_model")
        processor.save_pretrained(f"{CHECKPOINT_DIR}/best_model")
        print(f"  -> New best model saved (IoU: {best_val_iou:.4f})")

    # Early stopping: if no improvement for 8 epochs, stop
    if epoch > 8 and max(history["val_iou"][-8:]) < best_val_iou - 0.01:
        print(f"No improvement for 8 epochs - stopping early at epoch {epoch+1}")
        break

print(f"Training complete. Best validation IoU: {best_val_iou:.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history["val_iou"], color="green")
axes[1].axhline(y=0.65, color="red", linestyle="--", label="target (0.65)")
axes[1].set_title("Validation IoU")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{CHECKPOINT_DIR}/training_curves.png", dpi=150)
plt.show()

## 5. Final Evaluation on Held-Out Test Set

This is the real number — the test events were never seen during training OR validation.

In [ ]:
# Load the best checkpoint, not just whatever is in memory
best_model = SegformerForSemanticSegmentation.from_pretrained(
    f"{CHECKPOINT_DIR}/best_model"
).cuda()
best_model.eval()

test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

test_ious = []
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch["pixel_values"].cuda()
        labels = batch["labels"].cuda()

        outputs = best_model(pixel_values=pixel_values)
        preds = outputs.logits.argmax(dim=1)

        iou = compute_iou(preds, labels, ignore_index=IGNORE_INDEX)
        if iou is not None:
            test_ious.append(iou)

        valid = labels != IGNORE_INDEX
        all_preds.extend(preds[valid].cpu().numpy().flatten())
        all_labels.extend(labels[valid].cpu().numpy().flatten())

final_test_iou = sum(test_ious) / len(test_ious)
print(f"FINAL TEST IoU: {final_test_iou:.4f}")
print(f"Target was: 0.65")
print(f"Result: {'PASSED' if final_test_iou >= 0.65 else 'BELOW TARGET - consider more epochs or augmentation'}")

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

precision = precision_score(all_labels, all_preds, pos_label=1)
recall = recall_score(all_labels, all_preds, pos_label=1)
f1 = f1_score(all_labels, all_preds, pos_label=1)

print(f"Precision (of predicted flood, how much was correct): {precision:.4f}")
print(f"Recall (of actual flood, how much was caught):        {recall:.4f}")
print(f"F1 Score:                                              {f1:.4f}")

## 6. Visual Spot-Check on Test Tiles

Numbers can hide problems — always look at actual predictions before trusting the model.

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(12, 16))

for i in range(4):
    s2_path, lb_path = test_pairs[i]

    with rasterio.open(s2_path) as src:
        img_full = src.read(RGB_BANDS).astype(np.float32)
    with rasterio.open(lb_path) as src:
        label_full = src.read(1)

    img_display = np.clip(img_full / 3000.0, 0, 1).transpose(1, 2, 0)

    img_norm = np.clip(img_full / 3000.0 * 255.0, 0, 255).astype(np.uint8).transpose(1, 2, 0)
    encoded = processor(images=img_norm, return_tensors="pt")

    with torch.no_grad():
        output = best_model(pixel_values=encoded["pixel_values"].cuda())
        pred = output.logits.argmax(dim=1).squeeze().cpu().numpy()

    axes[i,0].imshow(img_display)
    axes[i,0].set_title(f"Input - {os.path.basename(s2_path)}")
    axes[i,1].imshow(label_full, cmap="Blues", vmin=-1, vmax=1)
    axes[i,1].set_title("Ground Truth")
    axes[i,2].imshow(pred, cmap="Blues", vmin=0, vmax=1)
    axes[i,2].set_title("Model Prediction")

plt.tight_layout()
plt.savefig(f"{CHECKPOINT_DIR}/test_predictions.png", dpi=150)
plt.show()

## 7. Push to HuggingFace Hub

Only run this once you are satisfied with the test IoU and visual predictions above.

In [ ]:
HF_REPO = "MuhammadBilal0021/segformer-flood-sentinel"

best_model.push_to_hub(HF_REPO)
processor.push_to_hub(HF_REPO)

print(f"Model pushed to: https://huggingface.co/{HF_REPO}")
print(f"Final test IoU: {final_test_iou:.4f}")
print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")
print("Codespaces can now load this model with:")
print(f'  SegformerForSemanticSegmentation.from_pretrained("{HF_REPO}")')